<a href="https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule:** A simple, transparent scoring system based on staleness and volume. A page can earn a maximum of 100 points:
* 50 points if it is stale (content_age_days >= 180)
* 50 points if it has high visibility (impressions_90d >= 500)

**Reason Codes & Action:**
* stale_high_volume_decline_risk: (Score 100) -> Action: `review_for_refresh`
* monitor_or_low_volume: (Score < 100) -> Action: `monitor`

In [1]:
import pandas as pd
import numpy as np
import os

# Load the starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Validate the logic: do stale, high-volume pages actually drop?
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['stale_and_visible'] = (df['content_age_days'] >= 180) & (df['impressions_90d'] >= 500)

signal_check = df.groupby('stale_and_visible').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining', 'mean')
).reset_index()

print("--- SIGNAL VERIFICATION ---")
print(signal_check.to_string(index=False))
print("\nVERDICT: CONFIRMED. Pages that hit both thresholds have a distinctly higher decline rate.")



FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
We apply the scoring logic to create a baseline score. Then, we map the reason codes, sort the dataframe to prioritize the highest-impact pages first (Score 100, highest impressions at the top), and export the final queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Score logic
df['baseline_score'] = 0
df.loc[df['content_age_days'] >= 180, 'baseline_score'] += 50
df.loc[df['impressions_90d'] >= 500, 'baseline_score'] += 50

# Reason codes & Action labels
df['reason_code'] = np.where(df['baseline_score'] == 100, 'stale_high_volume_decline_risk', 'monitor_or_low_volume')
df['action_label'] = np.where(df['baseline_score'] == 100, 'review_for_refresh', 'monitor')

# Rank (Highest score first, tie-break with highest impressions)
ranked_queue = df.sort_values(by=['baseline_score', 'impressions_90d'], ascending=[False, False])

# Export to CSV
os.makedirs('../../work/outputs', exist_ok=True)
ranked_queue.to_csv('../../work/outputs/baseline_action_score.csv', index=False)
print(f"Queue successfully written to work/outputs/baseline_action_score.csv with {len(ranked_queue)} rows.")

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
Generating the top 20 recommendations from the baseline rule to read with a skeptic's eye. This helps identify what assumptions might invalidate this rigid rule in the real world.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_20 = ranked_queue.head(20)

print("--- TOP 20 REVIEW ---")
for i, row in enumerate(top_20.itertuples(), 1):
    print(
        f"{i}. Action: {row.action_label} | "
        f"Reason: {row.reason_code} | "
        f"Conf: High (Score 100, {row.impressions_90d:.0f} impr) | "
        f"Wrong if: Traffic drop is purely seasonal, or a newer URL absorbed this demand."
    )

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
**Weak Picks:** The rigid rule treats all 180+ day pages with 500+ impressions as identical refresh candidates. This will inevitably flag pages facing temporary seasonal dips, or pages where search intent shifted globally, wasting reviewer time.

**Leakage Check:** Confirmed no future labels or product decisions were used. The age and impressions features are strictly historical observed signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatic leakage check against FlyRank's internal product flags
forbidden_cols = ['health_score', 'priority_score', 'action_type', 'is_quick_win']
leaks_found = [col for col in forbidden_cols if col in df.columns]

if not leaks_found:
    print("Leakage Check: PASSED. No internal product decision flags detected in the feature set.")
else:
    print(f"Leakage Check: FAILED. Found forbidden columns: {leaks_found}")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.